# Fine-tune Soprano TTS on Coral Danish

This notebook fine-tunes the Soprano TTS model on Danish speech using the Coral TTS dataset.

**Requirements:** A100 GPU (or similar with 40GB+ VRAM)

**Dataset:** [alexandrainst/coral-tts](https://huggingface.co/datasets/alexandrainst/coral-tts)

## 1. Setup Environment

In [ ]:
# Install dependencies
!pip install -q torch torchaudio transformers datasets huggingface_hub tqdm soundfile

In [ ]:
# Clone the soprano-factory repo with fixes
!git clone https://github.com/RJuro/soprano-factory.git
%cd soprano-factory
!git checkout claude/debug-coral-tts-danish-aMnnw

In [ ]:
# Verify GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Download and Prepare Coral TTS Danish Dataset

In [ ]:
from datasets import load_dataset
import os
import soundfile as sf
from tqdm import tqdm

# Configuration
MAX_SAMPLES = 2000  # Limit samples for testing

# Load Coral TTS dataset
print("Loading Coral TTS dataset...")
dataset = load_dataset("alexandrainst/coral-tts", split="train")
print(f"Full dataset: {len(dataset)} samples")
print(f"Using first {MAX_SAMPLES} samples for testing")

In [ ]:
# Inspect dataset structure
print("Dataset features:")
print(dataset.features)
print("\nExample sample:")
sample = dataset[0]
print(f"  speaker_id: {sample['speaker_id']}")
print(f"  transcription_id: {sample['transcription_id']}")
print(f"  text: {sample['text']}")
print(f"  audio: array shape {len(sample['audio']['array'])}, sr={sample['audio']['sampling_rate']} Hz")

In [ ]:
# Convert to LJSpeech format (limited to MAX_SAMPLES)
OUTPUT_DIR = "coral_danish_dataset"
WAVS_DIR = os.path.join(OUTPUT_DIR, "wavs")
os.makedirs(WAVS_DIR, exist_ok=True)

# Limit dataset size
num_samples = min(MAX_SAMPLES, len(dataset))
print(f"Converting {num_samples} samples to LJSpeech format...")

metadata_lines = []
for i, sample in enumerate(tqdm(dataset.select(range(num_samples)))):
    # Get audio and text (Coral TTS uses 'text' field)
    audio_array = sample['audio']['array']
    sample_rate = sample['audio']['sampling_rate']  # 44100 Hz, will be resampled to 32000
    text = sample['text']
    
    if not text or not text.strip():
        continue
    
    # Save audio file
    filename = f"coral_{i:06d}"
    audio_path = os.path.join(WAVS_DIR, f"{filename}.wav")
    sf.write(audio_path, audio_array, sample_rate)
    
    # Add to metadata (clean text of pipes)
    clean_text = text.replace('|', ' ').strip()
    metadata_lines.append(f"{filename}|{clean_text}")

# Write metadata file
metadata_path = os.path.join(OUTPUT_DIR, "metadata.txt")
with open(metadata_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(metadata_lines))

print(f"\nCreated {len(metadata_lines)} samples in {OUTPUT_DIR}/")
print(f"  - metadata.txt: {len(metadata_lines)} entries")
print(f"  - wavs/: {len(os.listdir(WAVS_DIR))} audio files")
print(f"  - Original sample rate: 44100 Hz (will be resampled to 32000 Hz)")

In [ ]:
# Preview some samples
print("First 5 metadata entries:")
for line in metadata_lines[:5]:
    filename, text = line.split('|', 1)
    print(f"  {filename}: {text[:80]}{'...' if len(text) > 80 else ''}")

## 3. Generate Audio Tokens

This step encodes all audio files into tokens using the Soprano encoder.

In [ ]:
# Generate the tokenized dataset
!python generate_dataset.py --input-dir coral_danish_dataset

In [ ]:
# Verify the generated files
import json

with open('coral_danish_dataset/train.json', 'r') as f:
    train_data = json.load(f)
with open('coral_danish_dataset/val.json', 'r') as f:
    val_data = json.load(f)

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"\nExample training sample:")
print(f"  Text: {train_data[0][0][:100]}...")
print(f"  Audio tokens: {len(train_data[0][1])} tokens")

## 4. Training Configuration

Adjusted for A100 GPU with batch size 48.

In [ ]:
# Training script with A100-optimized settings
TRAIN_SCRIPT = '''
"""
Training script for Soprano - A100 optimized.
Batch size 48 for A100 40GB/80GB.
"""
import argparse
import pathlib
import random
import time

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from dataset import AudioDataset


def get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input-dir",
        required=False,
        default="./coral_danish_dataset",
        type=pathlib.Path
    )
    parser.add_argument("--save-dir",
        required=True,
        type=pathlib.Path
    )
    parser.add_argument("--max-steps",
        required=False,
        default=10000,
        type=int
    )
    return parser.parse_args()

args = get_args()

# A100-optimized hyperparameters
device = \'cuda:0\'
seed = 1337
max_lr = 5e-4
warmup_ratio = 0.1
cooldown_ratio = 0.1
min_lr = 0.1 * max_lr
batch_size = 48  # Increased for A100
grad_accum_steps = 1
seq_len = 1024
val_freq = 250
text_factor = 0.01  # Small text loss for better text understanding
max_steps = args.max_steps
betas = (0.9, 0.95)
weight_decay = 0.1
train_dataset_path = f\'{args.input_dir}/train.json\'
val_dataset_path = f\'{args.input_dir}/val.json\'
save_path = args.save_dir

def worker_seed_init(_):
    worker_seed = torch.initial_seed() % (2**32-1)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def get_lr(it):
    if it<warmup_steps:
        return max_lr * (it+1) / warmup_steps
    if it<max_steps-cooldown_steps:
        return max_lr
    return min_lr + (max_lr-min_lr) * ((max_steps-it) / cooldown_steps)

def collate_pack(texts):
    tokens_batch = tokenizer(texts, padding=False, truncation=False)
    batch = []
    cur_sample, cur_size = [], 0
    for i in range(len(texts)):
        tokens = torch.tensor(tokens_batch[\'input_ids\'][i][:-1], dtype=torch.long)
        cur_size += tokens.size(0)
        cur_sample.append(tokens)
        if cur_size >= seq_len + 1:
            batch.append(torch.cat(cur_sample)[: seq_len + 1])
            cur_sample, cur_size = [], 0
            if len(batch) == batch_size:
                break
    if cur_sample and not batch:
        batch.append(torch.cat(cur_sample + [torch.zeros(seq_len, dtype=torch.long)])[: seq_len + 1])
    if len(batch) < batch_size:
        pad = batch[-1]
        while len(batch) < batch_size:
            batch.append(pad)
    batch = torch.stack(batch)
    x = batch[:, :-1]
    y = batch[:, 1:]
    return x, y

def compute_loss(logits, y, num_steps):
    pred = logits.view(-1, logits.size(-1))
    labels = y.reshape(-1)
    loss = torch.nn.functional.cross_entropy(pred, labels, reduction=\'none\')
    audio_mask = torch.logical_and(y>=3, y<=8003).view(-1)
    audio_loss = loss[audio_mask].mean()
    text_loss = loss[~audio_mask].mean()
    acc = (logits.argmax(dim=-1) == y).view(-1)[audio_mask].to(torch.float32).mean()
    audio_loss = audio_loss / num_steps
    text_loss = text_loss / num_steps
    acc = acc / num_steps
    return audio_loss, text_loss, acc

def evaluate(val_dataloader):
    model.eval()
    val_dataloader_it = iter(val_dataloader)
    with torch.no_grad():
        val_audio_loss_accum = torch.tensor(0.0).to(device)
        val_text_loss_accum = torch.tensor(0.0).to(device)
        val_acc_accum = torch.tensor(0.0).to(device)
        val_loss_steps = 1
        for _ in range(val_loss_steps):
            x, y = next(val_dataloader_it)
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device_type, dtype=torch.bfloat16):
                logits = model(x).logits
                audio_loss, text_loss, acc = compute_loss(logits, y, val_loss_steps)
            val_audio_loss_accum += audio_loss.detach()
            val_text_loss_accum += text_loss.detach()
            val_acc_accum += acc.detach()
        print(f"validation text loss: {val_text_loss_accum.item():.4f}\\tvalidation audio loss: {val_audio_loss_accum.item():.4f}\\tvalidation acc: {val_acc_accum.item():.4f}")
    model.train()


tokenizer = AutoTokenizer.from_pretrained(\'ekwek/Soprano-80M\')
if __name__ == \'__main__\':
    device_type = "cuda" if device.startswith("cuda") else "cpu"
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    torch.set_float32_matmul_precision(\'high\')
    print(f"Save Path: {save_path}")
    print(f"Batch Size: {batch_size}")
    print(f"Max Steps: {max_steps}")

    warmup_steps = int(max_steps * warmup_ratio)
    cooldown_steps = int(max_steps * cooldown_ratio)

    model = AutoModelForCausalLM.from_pretrained(\'ekwek/Soprano-80M\')
    model.to(torch.bfloat16).to(device)
    model.train()

    dataset = AudioDataset(train_dataset_path)
    dataloader = DataLoader(dataset,
        batch_size=batch_size * 16,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        worker_init_fn=worker_seed_init,
        collate_fn=collate_pack,
    )
    dataloader_it = iter(dataloader)
    val_dataset = AudioDataset(val_dataset_path)
    val_dataloader = DataLoader(val_dataset,
        batch_size=batch_size * 16,
        shuffle=False,
        num_workers=1,
        pin_memory=True,
        persistent_workers=True,
        worker_init_fn=worker_seed_init,
        collate_fn=collate_pack,
    )

    opt = torch.optim.AdamW(model.parameters(), max_lr, betas=betas, weight_decay=weight_decay, fused=True)

    pbar = tqdm(range(0, max_steps), ncols=200, dynamic_ncols=True)
    for step in pbar:
        start = time.time()
        if val_freq>0 and (step % val_freq == 0 or step==max_steps-1):
            evaluate(val_dataloader)

        opt.zero_grad()
        audio_loss_accum = 0.0
        text_loss_accum = 0.0
        acc_accum = 0.0
        for micro_step in range(grad_accum_steps):
            try:
                x, y = next(dataloader_it)
            except:
                dataloader_it = iter(dataloader)
                x, y = next(dataloader_it)
            x, y = x.to(device), y.to(device)

            with torch.autocast(device_type=device_type, dtype=torch.bfloat16):
                logits = model(x).logits
                audio_loss, text_loss, acc = compute_loss(logits, y, grad_accum_steps)
            audio_loss_accum += audio_loss.detach()
            text_loss_accum += text_loss.detach()
            acc_accum += acc.detach()
            total_loss = audio_loss + text_factor*text_loss
            total_loss.backward()

        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        lr = get_lr(step)
        for param_group in opt.param_groups:
            param_group[\'lr\'] = lr
        opt.step()
        torch.cuda.synchronize()
        total_tokens = step * batch_size*seq_len*grad_accum_steps
        end = time.time()
        dt = (end-start)*1000
        tokens_per_second = (batch_size*seq_len*grad_accum_steps) / (end-start)
        tqdm_log = f\'text loss: {text_loss_accum.item():.3f} | audio loss: {audio_loss_accum.item():.3f} | acc: {acc_accum.item():.4f} | lr: {lr:.2e} | norm: {norm:.3f} | time: {dt:.2f} ms | {tokens_per_second:.2f} t/s\'
        pbar.set_description(tqdm_log)

        # Save checkpoint every 1000 steps
        if (step + 1) % 1000 == 0:
            checkpoint_path = f"{save_path}_step{step+1}"
            print(f"\\nSaving checkpoint at step {step+1}...")
            model.save_pretrained(checkpoint_path)
            tokenizer.save_pretrained(checkpoint_path)

    print(f"Training complete. Saving model at {save_path}")
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print("Saving done.")
'''

with open('train_a100.py', 'w') as f:
    f.write(TRAIN_SCRIPT)
print("Created train_a100.py with A100-optimized settings")

## 5. Start Training

In [ ]:
# Create output directory
!mkdir -p outputs/soprano-danish

In [ ]:
# Start training (2000 steps for 2000 samples test run)
!python train_a100.py --save-dir outputs/soprano-danish --max-steps 2000

## 6. Test the Fine-tuned Model

In [ ]:
# Test inference with the fine-tuned model
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load fine-tuned model
model_path = "outputs/soprano-danish"
model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)
model.to('cuda').to(torch.bfloat16)
model.eval()

print("Model loaded successfully!")

In [ ]:
# Generate audio tokens from Danish text
def generate_audio_tokens(text, max_new_tokens=500):
    prompt = f"[STOP][TEXT]{text}[START]"
    inputs = tokenizer(prompt, return_tensors="pt").to('cuda')
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.encode('[STOP]')[0]
        )
    
    generated = tokenizer.decode(outputs[0])
    return generated

# Test with Danish text
test_texts = [
    "Hej, mit navn er Soprano.",
    "Rød grød med fløde er en dansk dessert.",
    "København er Danmarks hovedstad."
]

for text in test_texts:
    print(f"\nInput: {text}")
    result = generate_audio_tokens(text)
    # Count audio tokens generated
    import re
    audio_tokens = re.findall(r'\[\d+\]', result)
    print(f"Generated {len(audio_tokens)} audio tokens")

## 7. Save Model to Hugging Face Hub (Optional)

In [ ]:
# Uncomment and run to upload to HuggingFace
# from huggingface_hub import login, HfApi
# login()  # Enter your HF token
# 
# model.push_to_hub("your-username/soprano-danish")
# tokenizer.push_to_hub("your-username/soprano-danish")

## Notes

- **Sample limit:** Currently set to 2000 samples for testing. Change `MAX_SAMPLES` in cell 2.1 for full dataset (~18k samples)
- **Batch size 48** is optimized for A100 40GB. If you have A100 80GB, you can try batch size 64-96
- **Training steps:** 2000 steps for test run. Increase `--max-steps` for better results
- **Checkpoints** are saved every 1000 steps
- **Audio resampling:** Coral TTS is 44100 Hz, automatically resampled to 32000 Hz
- If you run out of memory, reduce batch_size to 32 or 24